# Error Analysis and Explainability

This notebook loads saved benchmark predictions, identifies model errors, and generates token-level explanations for selected examples.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.evaluation_runs import load_latest_evaluation_artifacts
from utils.model_loader import load_fake_news_model
from utils.prediction import clean_special_tokens, get_word_attributions, merge_wordpiece_tokens, top_k_attributions
from utils.project_config import ARTIFACTS_DIR

pd.set_option("display.max_colwidth", 180)

## Load Latest Fake-News Benchmark Artifacts

In [ ]:
metrics, predictions, run = load_latest_evaluation_artifacts("fake_news", "fake-news-kaggle")
run.run_dir, metrics["accuracy"], metrics["macro_f1"]

## Identify Error Groups

In [ ]:
errors = predictions.loc[~predictions["is_correct"]].copy()
correct = predictions.loc[predictions["is_correct"]].copy()

false_positives = errors.loc[(errors["true_label"] == "REAL") & (errors["predicted_label"] == "FAKE")]
false_negatives = errors.loc[(errors["true_label"] == "FAKE") & (errors["predicted_label"] == "REAL")]
high_confidence_errors = errors.sort_values("confidence", ascending=False).head(10)
low_confidence_predictions = predictions.sort_values("confidence", ascending=True).head(10)

{
    "errors": len(errors),
    "false_positives": len(false_positives),
    "false_negatives": len(false_negatives),
    "correct": len(correct),
}

In [ ]:
high_confidence_errors[["true_label", "predicted_label", "confidence", "title"]].head(10)

In [ ]:
low_confidence_predictions[["true_label", "predicted_label", "confidence", "title"]].head(10)

## Save Representative Error Examples

In [ ]:
output_dir = ARTIFACTS_DIR / "error_analysis" / "fake_news"
output_dir.mkdir(parents=True, exist_ok=True)

example_columns = [
    "title",
    "true_label",
    "predicted_label",
    "confidence",
    "prob_fake",
    "prob_real",
    "input_text",
]

false_positives.head(10)[example_columns].to_csv(output_dir / "false_positives.csv", index=False)
false_negatives.head(10)[example_columns].to_csv(output_dir / "false_negatives.csv", index=False)
high_confidence_errors.head(10)[example_columns].to_csv(output_dir / "high_confidence_errors.csv", index=False)
low_confidence_predictions.head(10)[example_columns].to_csv(output_dir / "low_confidence_predictions.csv", index=False)

output_dir

## Token-Level Explanations

The helper below uses `transformers-interpret`. Start with a small number of examples because attribution can be slow.

In [ ]:
model, tokenizer = load_fake_news_model()

examples_to_explain = pd.concat([
    correct.sort_values("confidence", ascending=False).head(3),
    high_confidence_errors.head(3),
], ignore_index=True)

explanation_rows = []
for index, row in examples_to_explain.iterrows():
    attributions = get_word_attributions(model, tokenizer, row["input_text"])
    if attributions and attributions[0][0] != "ERROR":
        attributions = clean_special_tokens(merge_wordpiece_tokens(attributions))
        top_terms = top_k_attributions(attributions, k=20)
    else:
        top_terms = attributions

    explanation_rows.append({
        "example_index": index,
        "true_label": row["true_label"],
        "predicted_label": row["predicted_label"],
        "confidence": row["confidence"],
        "title": row.get("title", ""),
        "top_terms": top_terms,
    })

explanations = pd.DataFrame(explanation_rows)
explanations

In [ ]:
explainability_dir = ARTIFACTS_DIR / "explainability"
explainability_dir.mkdir(parents=True, exist_ok=True)
explanations.to_json(explainability_dir / "fake_news_explanation_examples.json", orient="records", indent=2)
explainability_dir

## Interpretation Notes

Use this section to summarize whether the highlighted tokens look meaningful or misleading. Treat the attributions as a model-behavior diagnostic, not as evidence that a claim is true or false.